In [1]:
using ITensors
using ITensorMPS
using LinearAlgebra

In [3]:
include("../QCSB/QCSB.jl")
include("../src/circuit.jl")

sample_entropy_circuit (generic function with 1 method)

In [11]:
function ferromagnetic_state(::Type{DiagonalStateMPS}, L::Int; ref=false)
    N = L+ref
    sites = siteinds("Qubit", N)
    ψ0 = MPS(sites, _ -> "0")
    ψ1 = MPS(sites, _ -> "1")
    return DiagonalStateMPS((ψ0 + ψ1)/2)
end


ferromagnetic_state (generic function with 1 method)

In [25]:
function drop_small(M::AbstractMatrix; cutoff=1e-8)
    M[abs.(M) .< cutoff] .= 0
    return M
end


drop_small (generic function with 1 method)

In [ ]:
L = 10
ψ = zero_state(DiagonalStateMPS, L)

Xn = decoherence_layer(ψ, PauliX, 0.5, 1:L)
ψ = apply(Xn, ψ)
ψ = normalize(ψ)
truncate!(ψ; cutoff=1e-8, maxdim=200)
# real.(dense(state))

DiagonalStateMPS(MPS(10))

In [50]:
correlator(state.mps, ("Z", "Z", "Z", "Z"), [(i,j,k,l) for i in 1:L for j in 1:L for k in 1:L for l in 1:L]);

In [49]:
for i in 1:L, j in 1:L, k in 1:L, l in 1:L
    x = correlator(state.mps, ("Z", "Z", "Z", "Z"), [(i,j,k,l)])
end

In [28]:
L = 3
state = ferromagnetic_state(DiagonalStateMPS, L)

Xn = decoherence_layer(state, PauliX, 0.5, 1:L)
state = apply(Xn, state)
state = normalize(state)
truncate!(state; cutoff=1e-8, maxdim=200)
real.(dense(state))

8×8 SparseMatrixCSC{Float64, Int64} with 8 stored entries:
 0.125   ⋅      ⋅      ⋅      ⋅      ⋅      ⋅      ⋅ 
  ⋅     0.125   ⋅      ⋅      ⋅      ⋅      ⋅      ⋅ 
  ⋅      ⋅     0.125   ⋅      ⋅      ⋅      ⋅      ⋅ 
  ⋅      ⋅      ⋅     0.125   ⋅      ⋅      ⋅      ⋅ 
  ⋅      ⋅      ⋅      ⋅     0.125   ⋅      ⋅      ⋅ 
  ⋅      ⋅      ⋅      ⋅      ⋅     0.125   ⋅      ⋅ 
  ⋅      ⋅      ⋅      ⋅      ⋅      ⋅     0.125   ⋅ 
  ⋅      ⋅      ⋅      ⋅      ⋅      ⋅      ⋅     0.125

In [29]:
L = 3
state = ghz_state(MixedStateMPS, L)

Xn = decoherence_layer(state, PauliX, 0.5, 1:L)
state = apply(Xn, state)
state /= norm(state)
truncate!(state; cutoff=1e-8, maxdim=200)
real.(drop_small(dense(state)))

8×8 Matrix{Float64}:
 0.125  0.0    0.0    0.0    0.0    0.0    0.0    0.125
 0.0    0.125  0.0    0.0    0.0    0.0    0.125  0.0
 0.0    0.0    0.125  0.0    0.0    0.125  0.0    0.0
 0.0    0.0    0.0    0.125  0.125  0.0    0.0    0.0
 0.0    0.0    0.0    0.125  0.125  0.0    0.0    0.0
 0.0    0.0    0.125  0.0    0.0    0.125  0.0    0.0
 0.0    0.125  0.0    0.0    0.0    0.0    0.125  0.0
 0.125  0.0    0.0    0.0    0.0    0.0    0.0    0.125

In [ ]:
function binder_EA(state::DiagonalStateMPS, L::Int)
    quads = [(i,j,k,l) for i in 1:L for j in i+1:L for k in j+1:L for l in k+1:L]
    quad_corrs = correlator(state.mps, ("Z", "Z", "Z", "Z"), quads)

    pair_corrs = correlation_matrix()

    numerator = sum(quad_corrs) - 2*sum(pair_corrs) + L

    EA = 0
    for i in 1:L, j in 1:L, k in 1:L, l in 1:L
        EA += correlator(state.mps, ("Z", "Z", "Z", "Z"), (i,j,k,l))
    end
    return EA / L^4
end

In [36]:
using ITensorCorrelators

In [ ]:
correlator(state.mps, ("X", "X", "X", "X"), [(i,j,k,l) for i in 1:L for j in 1:L for k in 1:L for l in 1:L])

Dict{Tuple{Vararg{Int64}}, ComplexF64} with 81 entries:
  (3, 2, 1, 3) => 2.31235e-32+0.0im
  (2, 2, 2, 1) => 2.31235e-32+0.0im
  (1, 2, 1, 2) => 0.125+0.0im
  (3, 1, 1, 2) => 9.92841e-33+0.0im
  (1, 2, 3, 1) => 9.92841e-33+0.0im
  (3, 1, 3, 1) => 0.125+0.0im
  (2, 1, 3, 3) => 2.31235e-32+0.0im
  (3, 2, 1, 2) => 3.18392e-32+0.0im
  (3, 2, 3, 1) => 2.31235e-32+0.0im
  (2, 1, 1, 1) => 2.31235e-32+0.0im
  (2, 2, 3, 3) => 0.125+0.0im
  (1, 3, 2, 1) => 9.92841e-33+0.0im
  (2, 1, 3, 2) => 3.18392e-32+0.0im
  (2, 2, 1, 1) => 0.125+0.0im
  (3, 3, 2, 1) => 2.31235e-32+0.0im
  (2, 2, 3, 2) => 1.59196e-32+0.0im
  (2, 3, 2, 3) => 0.125+0.0im
  (1, 3, 3, 3) => 3.18392e-32+0.0im
  (2, 3, 2, 2) => 1.59196e-32+0.0im
  ⋮            => ⋮

In [40]:
[(i,j,k,l) for i in 1:L for j in 1:L for k in 1:L for l in 1:L]

81-element Vector{NTuple{4, Int64}}:
 (1, 1, 1, 1)
 (1, 1, 1, 2)
 (1, 1, 1, 3)
 (1, 1, 2, 1)
 (1, 1, 2, 2)
 (1, 1, 2, 3)
 (1, 1, 3, 1)
 (1, 1, 3, 2)
 (1, 1, 3, 3)
 (1, 2, 1, 1)
 ⋮
 (3, 3, 1, 1)
 (3, 3, 1, 2)
 (3, 3, 1, 3)
 (3, 3, 2, 1)
 (3, 3, 2, 2)
 (3, 3, 2, 3)
 (3, 3, 3, 1)
 (3, 3, 3, 2)
 (3, 3, 3, 3)

In [ ]:
function circuit(L::Int, )